# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [6]:
!pip install -U "bitsandbytes>=0.46.1" transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 36.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0
  Attempting uninstall: peft
    Found existing installation: peft 0.21.0
    Uninstalling peft-0.21.0:
      Successfully uninstalled peft-0.21.0


In [1]:
# YOUR CODE HERE
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Must match Notebook 2
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully.")
print("Model:", MODEL_ID)
print("Device:", model.device)

query = (
    "My package was supposed to arrive a few days ago, "
    "but I still haven't received it. What is going on?"
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a customer support assistant. "
            "Answer the customer's question clearly and helpfully."
        )
    },
    {
        "role": "user",
        "content": query
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)



model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
<|im_start|>system
You are a customer support assistant. Answer the customer's question clearly and helpfully.<|im_end|>
<|im_start|>user
My package was supposed to arrive a few days ago, but I still haven't received it. What is going on?<|im_end|>
<|im_start|>assistant



In [2]:
# YOUR CODE HERE
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id
    )

# Remove the original prompt from the generated sequence
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Customer Query:")
print(query)

print("\nBaseline Model Response:")
print(response)

baseline_result = {
    "model": MODEL_ID,
    "query": query,
    "response": response
}

import json

with open("baseline_result.json", "w", encoding="utf-8") as f:
    json.dump(baseline_result, f, indent=4, ensure_ascii=False)

print("Baseline result saved to baseline_result.json")

Customer Query:
My package was supposed to arrive a few days ago, but I still haven't received it. What is going on?

Baseline Model Response:
I'm sorry to hear that you're experiencing delays with your package delivery. Here are some steps you can take:

1. Check your tracking information: Make sure you have access to the tracking number for your package. This will allow you to monitor its progress.

2. Contact the shipping company: Reach out to the shipping company responsible for delivering your package (e.g., UPS, FedEx, or USPS) to inquire about any issues they might be facing.

3. Verify the status of your order: If possible, check if your order has been canceled due to unforeseen circumstances like weather conditions or other
Baseline result saved to baseline_result.json


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
# YOUR CODE HERE
ground_truth_rule = (
    "Domestic orders deliver within 3-7 business days."
)

print("Ground-truth SOP rule:")
print(ground_truth_rule)

print("Customer Query:")
print(query)

print("\nBaseline Response:")
print(response)

print("\nGround-truth SOP:")
print(ground_truth_rule)

assessment = {
    "Ground_truth_rule": ground_truth_rule,
    "Invented_delivery_timeline": "Check whether the model gives a delivery period different from 3-7 business days.",
    "Invented_tracking_system": "Check whether the model mentions a tracking system that is not stated in the SOP.",
    "Invented_department": "Check whether the model refers to a support/logistics department that is not stated in the SOP.",
    "Unsupported_claims": "Check for any other factual claims not supported by the SOP."
}

for item, description in assessment.items():
    print(f"{item}: {description}")

baseline_findings = {
    "SOP_rule": "Domestic orders deliver within 3-7 business days.",
    "Invented_delivery_timeline": "Yes/No",
    "Invented_tracking_system": "Yes/No",
    "Invented_department": "Yes/No",
    "Other_hallucinations": "None / describe the unsupported claim",
    "Overall_assessment": "Accurate / Contains unsupported information"
}

print("\nBaseline Assessment:")
for key, value in baseline_findings.items():
    print(f"{key}: {value}")

Ground-truth SOP rule:
Domestic orders deliver within 3-7 business days.
Customer Query:
My package was supposed to arrive a few days ago, but I still haven't received it. What is going on?

Baseline Response:
I'm sorry to hear that you're experiencing delays with your package delivery. Here are some steps you can take:

1. Check your tracking information: Make sure you have access to the tracking number for your package. This will allow you to monitor its progress.

2. Contact the shipping company: Reach out to the shipping company responsible for delivering your package (e.g., UPS, FedEx, or USPS) to inquire about any issues they might be facing.

3. Verify the status of your order: If possible, check if your order has been canceled due to unforeseen circumstances like weather conditions or other

Ground-truth SOP:
Domestic orders deliver within 3-7 business days.
Ground_truth_rule: Domestic orders deliver within 3-7 business days.
Invented_delivery_timeline: Check whether the model 

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [5]:
# YOUR CODE HERE
import json
import os

baseline_artifact = {
    "model_id": MODEL_ID,
    "query": query,
    "response": response,
    "ground_truth_rule": ground_truth_rule
}

with open("baseline_output.json", "w", encoding="utf-8") as f:
    json.dump(
        baseline_artifact,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Baseline artifact saved successfully.")
print("File: baseline_output.json")

if os.path.exists("baseline_output.json"):
    print("✓ baseline_output.json exists")

    with open("baseline_output.json", "r", encoding="utf-8") as f:
        saved_baseline = json.load(f)

    print("\nSaved baseline:")
    print(json.dumps(saved_baseline, indent=4, ensure_ascii=False))
else:
    print("✗ baseline_output.json was not created")

with open("baseline_output.json", "r", encoding="utf-8") as f:
    baseline = json.load(f)

print(baseline["query"])
print(baseline["response"])

Baseline artifact saved successfully.
File: baseline_output.json
✓ baseline_output.json exists

Saved baseline:
{
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "query": "My package was supposed to arrive a few days ago, but I still haven't received it. What is going on?",
    "response": "I'm sorry to hear that you're experiencing delays with your package delivery. Here are some steps you can take:\n\n1. Check your tracking information: Make sure you have access to the tracking number for your package. This will allow you to monitor its progress.\n\n2. Contact the shipping company: Reach out to the shipping company responsible for delivering your package (e.g., UPS, FedEx, or USPS) to inquire about any issues they might be facing.\n\n3. Verify the status of your order: If possible, check if your order has been canceled due to unforeseen circumstances like weather conditions or other",
    "ground_truth_rule": "Domestic orders deliver within 3-7 business days."
}
My package was supp

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**